# Example on How to load Qwen Model with Hugging Face Transformers Library on Kaggle Workspace

Model used: [Qwen3-30B-A3B](https://huggingface.co/Qwen/Qwen3-30B-A3B)

NOTE: Expect recurring bugs if loading transformers with [Qwen3-32B-FP8 Quant Type](https://huggingface.co/Qwen/Qwen3-32B-FP8). Issues resolve with restarting the Jupyter Kernel but the problem will recur immediately after first run


In [2]:
!pip install transformers accelerate torch

In [3]:
import torch 

assert torch.cuda.is_available(), "No GPU detected — enable GPU in Kaggle: Settings → Accelerator → GPU T4 x2"
print(f"GPUs available : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")

display(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

GPUs available : 2
  GPU 0: Tesla T4 | 15.64 GB
  GPU 1: Tesla T4 | 15.64 GB


'VRAM free: 15.5 GB'

In [5]:
# Load the Model with Hugging Face
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from transformers import AutoModelForCausalLM, AutoTokenizer

# parent model name = "Qwen/Qwen3-30B-A3B"
MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            dtype=torch.bfloat16,   
            local_files_only=True,
        )

model.to("cuda:0")

display(f"Completed loading model from path {model_path}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

'Completed loading model from path /kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1'

In [8]:
# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
display(f"Completed preparing model inputs: {model_inputs}")

"Completed preparing model inputs: {'input_ids': tensor([[151644,    872,    198,  35127,    752,    264,   2805,  16800,    311,\n           3460,   4128,   1614,     13, 151645,    198, 151644,  77091,    198]],\n       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],\n       device='cuda:0')}"

In [ ]:
# conduct text completion
model.eval()

with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=False,            
        pad_token_id=tokenizer.eos_token_id,
    )

#output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

display(f"Completed Text Completion: \nComplete Output: {generated_ids}")

"Completed Text Completion: \nComplete Output: tensor([[151644,    872,    198,  35127,    752,    264,   2805,  16800,    311,\n           3460,   4128,   1614,     13, 151645,    198, 151644,  77091,    198,\n          34253,   4128,   4119,    320,   4086,  21634,      8,    525,    264,\n            943,    315,  20443,  11229,    320,  15469,      8,    429,    646,\n           6923,   3738,  12681,   1467,   3118,    389,    279,   1946,    807,\n           5258,     13,   2379,    525,  16176,    389,  12767,  14713,    315,\n           1467,    821,     11,   1741,    438,   6467,     11,   9709,     11,\n            323,  13037,     11,    323,    646,   3535,    323,   6923,   1467,\n            304,    264,   8045,    315,  15459,    323,   9222,     13,    444,\n          10994,     82,    525,   1483,    304,    264,   6884,   2088,    315,\n           8357,     11,   2670,   6236,  61905,     11,   4108,  56519,     11,\n            323,   2213,   9471,     13,   2379,   

In [ ]:
# parsing thinking content

output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


thinking content: 
content: Large language models (LLMs) are a type of artificial intelligence (AI) that can generate human-like text based on the input they receive. They are trained on vast amounts of text data, such as books, articles, and websites, and can understand and generate text in a variety of languages and styles. LLMs are used in a wide range of applications, including chatbots, virtual assistants, and content generation. They are becoming increasingly popular as they can provide a more natural and engaging user experience than traditional rule-based systems.


In [ ]:
#from transformers import GenerationConfig
#?GenerationConfig

Init signature: GenerationConfig(**kwargs)
Docstring:     
Class that holds a configuration for a generation task. A `generate` call supports the following generation methods
for text-decoder, text-to-text, speech-to-text, and vision-to-text models:

    - *greedy decoding* if `num_beams=1` and `do_sample=False`
    - *multinomial sampling* if `num_beams=1` and `do_sample=True`
    - *beam-search decoding* if `num_beams>1` and `do_sample=False`
    - *beam-search multinomial sampling* if `num_beams>1` and `do_sample=True`
    - *assisted decoding* if `assistant_model` or `prompt_lookup_num_tokens` is passed to `.generate()`

To learn more about decoding strategies refer to the [text generation strategies guide](../generation_strategies).

<Tip>

A large number of these flags control the logits or the stopping criteria of the generation. Make sure you check
the [generate-related classes](https://huggingface.co/docs/transformers/internal/generation_utils) for a full
description of the po